# 04. System-Flavoured Designs: Rate Limiter, Consistent Hashing, Bloom Filter

Modern interviews often mix DSA with **system design thinking**. These three small structures show up in real backends (API gateways, caches, databases) and are easy to code.

**Learning order:** RATE LIMITER ALGORITHMS → TOKEN BUCKET → CONSISTENT HASHING → BLOOM FILTER → HOW TO APPROACH DESIGN ROUNDS → PRACTICE MAP

## 1. Rate limiter algorithms

Limit a user to `N` requests per time window.

| Algorithm | Idea | Pros / cons |
|---|---|---|
| **Fixed window counter** | count requests in each minute | simple, but a burst at the boundary can double the limit |
| **Sliding window log** | keep timestamps of recent requests (deque) | exact, memory per request |
| **Token bucket** | a bucket refills at a steady rate; each request takes a token | allows **bursts** up to capacity, smooth average rate, O(1) memory |
| **Leaky bucket** | requests queue and leave at a fixed rate | smooth output rate |

In [1]:
class TokenBucket:
    def __init__(self, capacity, refill_rate, clock):
        self.capacity, self.rate, self.clock = capacity, refill_rate, clock
        self.tokens = capacity
        self.last = clock()

    def allow(self):
        now = self.clock()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)   # refill
        self.last = now
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

t = [0.0]
tb = TokenBucket(capacity=2, refill_rate=1, clock=lambda: t[0])     # 1 token per second
print([tb.allow() for _ in range(3)])   # [True, True, False]
t[0] = 1.0
print(tb.allow(), tb.allow())           # True False
t[0] = 10.0                             # long pause: the bucket refills only up to capacity
print([tb.allow() for _ in range(3)])   # [True, True, False]

[True, True, False]
True False
[True, True, False]


## 2. Consistent hashing

**Problem:** with `hash(key) % N` servers, adding or removing a server **remaps almost every key** (the whole cache goes cold). **Consistent hashing** places servers and keys on a **ring** (hash space). A key belongs to the **first server clockwise** from its position. Removing a server moves **only that server's keys** to its neighbour.

**Virtual nodes:** each server appears at many ring positions so load spreads evenly.

Used in: distributed caches (Memcached clients), databases (Cassandra, DynamoDB), load balancers.

In [2]:
import hashlib
from bisect import bisect

class ConsistentHashRing:
    def __init__(self, replicas=50):
        self.replicas = replicas
        self.ring = {}                                       # hash -> server
        self.sorted_hashes = []

    @staticmethod
    def _hash(value):
        return int(hashlib.md5(value.encode()).hexdigest(), 16)

    def add_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f"{node}#{i}")
            self.ring[h] = node
            self.sorted_hashes.append(h)
        self.sorted_hashes.sort()

    def remove_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f"{node}#{i}")
            del self.ring[h]
            self.sorted_hashes.remove(h)

    def get_node(self, key):
        i = bisect(self.sorted_hashes, self._hash(key)) % len(self.sorted_hashes)   # first clockwise
        return self.ring[self.sorted_hashes[i]]

ring = ConsistentHashRing()
for n in ["A", "B", "C"]:
    ring.add_node(n)
keys = [f"key{i}" for i in range(1000)]
before = {k: ring.get_node(k) for k in keys}
ring.remove_node("B")
after = {k: ring.get_node(k) for k in keys}
moved = [k for k in keys if before[k] != after[k]]
print(all(before[k] == "B" for k in moved), len(moved) < 600)   # True True

True True


## 3. Bloom filter

A **space-efficient set** that can say:
- "**definitely not** present" (always correct), or
- "**probably present**" (may be a **false positive**).

It uses a bit array and `k` hash functions. `add(x)` sets `k` bits. `might_contain(x)` checks that **all** `k` bits are set. It never has false negatives and never stores the items.

Used in: databases (skip disk reads for missing keys), web crawlers (seen URLs), malware/URL blocklists.

In [3]:
class BloomFilter:
    def __init__(self, size=10000, hashes=3):
        self.size, self.hashes = size, hashes
        self.bits = bytearray(size)

    def _positions(self, item):
        for i in range(self.hashes):
            h = hashlib.sha256(f"{i}:{item}".encode()).hexdigest()
            yield int(h, 16) % self.size

    def add(self, item):
        for p in self._positions(item):
            self.bits[p] = 1

    def might_contain(self, item):
        return all(self.bits[p] for p in self._positions(item))

bf = BloomFilter()
words = [f"user{i}" for i in range(500)]
for w in words:
    bf.add(w)
print(all(bf.might_contain(w) for w in words))   # True  (no false negatives)
false_positives = sum(bf.might_contain(f"other{i}") for i in range(1000))
print(false_positives < 100)   # True  (a few false positives are possible)

True
True


## 4. How to approach a design round (low-level vs high-level)

**Low-level design (this topic):** classes and data structures. Clarify the API, choose structures, state complexities, write clean code, discuss thread-safety briefly.

**High-level / system design:** services and scale. A usable structure:
1. **Requirements:** functional, non-functional (QPS, latency, availability, consistency), constraints.
2. **Estimation:** users, requests per second, storage.
3. **API and data model.**
4. **High-level diagram:** clients, load balancer, services, cache, database, queue.
5. **Deep dive:** the hardest part (cache invalidation, sharding with consistent hashing, rate limiting).
6. **Trade-offs:** consistency vs availability, cost, failure modes.

Knowing the **data structures behind these systems** (hash rings, LRU, Bloom filters, heaps, tries, queues) is exactly why this DSA roadmap matters.

## 5. Interview thinking

**Say:** "Fixed window is simple but allows boundary bursts. I'd use a token bucket: O(1) memory per user, supports bursts, smooth average rate."

**Common mistakes:** forgetting thread safety (locks or atomic operations in production), using the wall clock without injecting it (untestable), `hash(key) % N` when servers change, believing a Bloom filter can delete items or has no false positives.

## 6. Practice map (for later)

| Level | Problems |
|---|---|
| Basic | Design HashMap (706), Design HashSet (705), Min Stack (155), Logger Rate Limiter (359), Moving Average from Data Stream (346) |
| Pattern | Insert Delete GetRandom O(1) (380), Time Based Key-Value Store (981), LRU Cache (146), Design Browser History (1472), Flatten Nested List Iterator (341) |
| Standard | LFU Cache (460), Snapshot Array (1146), Design Twitter (355), Design Underground System (1396), Peeking Iterator (284), Design Hit Counter (362) |
| Hard | All O`one Data Structure (432), Design Search Autocomplete System (642), Design In-Memory File System (588), Max Stack (716) |

## Quick Revision

- Rate limiter: token bucket = burst + steady rate. Inject the clock.
- Consistent hashing: ring + virtual nodes, only the removed node's keys move.
- Bloom filter: no false negatives, some false positives, no deletes.
- Design answer = API, structures per operation, invariants, complexity.